# Notebook 1: Initial Baseline Training & Minority Collapse (A0)
## Class-Sensitive Deep Active Learning (CSDA-AL) on ISIC 2019

[![Open In Colab](https://colab.research.google.com/assets/colab-badge.svg)](https://colab.research.google.com/github/Dishitha09/csda-al/blob/main/notebooks/01_Initial_Baseline_A0_Training.ipynb)

**Author:** Group 10 | Dishitha Ramesh | Final ML Course Project  
**Stage:** Baseline Milestone 1 (Where the project started)  
**Dataset:** ISIC 2019 (25,331 clinical images, 8 diagnostic classes; 53.9:1 severe imbalance ratio)  
**Locked Test Set:** 3,799 held-out images (Canonical 70/15/15 Partition)

---

### 1. Environment & Repository Setup
Ensures the workspace is set up and dependencies are installed.

In [ ]:
import os, sys
if not os.path.exists('src'):
    !git clone https://github.com/Dishitha09/csda-al.git
    %cd csda-al

sys.path.insert(0, os.path.abspath('src'))
!pip install -q scikit-learn pandas matplotlib reportlab
print('Environment configured successfully!')

### 2. Load Canonical Data Splits & Cached Features
Loads the cryptographically disjoint feature tensors extracted from the ResNet-18 backbone.

In [ ]:
import torch
import numpy as np
import pandas as pd

data_dir = 'data/cached_features'
train_f = torch.load(f'{data_dir}/train_features.pt', map_location='cpu', weights_only=False)
val_f = torch.load(f'{data_dir}/val_features.pt', map_location='cpu', weights_only=False)
test_f = torch.load(f'{data_dir}/test_features.pt', map_location='cpu', weights_only=False)

X_train_full = train_f['features']
y_train_full = train_f['labels']
X_test = test_f['features']
y_test = test_f['labels'].numpy()

CLASSES = ['MEL', 'NV', 'BCC', 'AK', 'BKL', 'DF', 'VASC', 'SCC']
print(f'Train pool: {len(X_train_full):,} samples')
print(f'Held-out Test: {len(X_test):,} samples (100% disjoint)')
print('Class distribution in Test Set:', dict(zip(CLASSES, np.bincount(y_test, minlength=8))))

### 3. Train Initial Baseline Model (A0: Unweighted CE + Frozen Linear Head)
Simulates the initial state: training with standard unweighted Cross-Entropy loss at budget B=320.

In [ ]:
import torch.nn as nn
import torch.optim as optim

# Linear classification probe on frozen representations
class LinearProbe(nn.Module):
    def __init__(self, in_features=512, num_classes=8):
        super().__init__()
        self.fc = nn.Linear(in_features, num_classes)
    def forward(self, x):
        return self.fc(x)

# Sample initial budget B=320 using random seed 42
rng = np.random.RandomState(42)
init_indices = rng.choice(len(X_train_full), size=320, replace=False)
X_init = X_train_full[init_indices]
y_init = y_train_full[init_indices]

# Train with unweighted standard cross-entropy loss
torch.manual_seed(42)
model_a0 = LinearProbe(512, 8)
criterion = nn.CrossEntropyLoss() # Unweighted baseline
optimizer = optim.AdamW(model_a0.parameters(), lr=0.01, weight_decay=1e-4)

model_a0.train()
for epoch in range(40):
    optimizer.zero_grad()
    out = model_a0(X_init)
    loss = criterion(out, y_init)
    loss.backward()
    optimizer.step()

print('Baseline Model A0 successfully trained on 320 images!')

### 4. Evaluate Initial Baseline on Locked Test Set (3,799 Images)
Evaluates discrete predictions using standard argmax.

In [ ]:
from sklearn.metrics import balanced_accuracy_score, roc_auc_score, f1_score, confusion_matrix, recall_score

model_a0.eval()
with torch.no_grad():
    logits = model_a0(X_test)
    probs = torch.softmax(logits, dim=1).numpy()
    preds = np.argmax(probs, axis=1)

ba = balanced_accuracy_score(y_test, preds) * 100
f1 = f1_score(y_test, preds, average='macro')
auroc = roc_auc_score(pd.get_dummies(y_test), probs, average='macro')
recalls = recall_score(y_test, preds, average=None) * 100
min_rec = float((recalls[5] + recalls[6] + recalls[7]) / 3.0)

print('=' * 75)
print('             INITIAL BASELINE (A0) TEST SET PERFORMANCE')
print('=' * 75)
print(f'  • Balanced Accuracy:        {ba:.2f}% (Arithmetic mean of 8 recalls)')
print(f'  • Macro-AUROC:              {auroc:.4f}')
print(f'  • Macro-F1 Score:           {f1:.4f}')
print(f'  • Minority Recall (DF/VASC): {min_rec:.2f}% (CRITICAL FAILURE)')
print('=' * 75)

df_recs = pd.DataFrame({
    'Diagnosis': CLASSES,
    'Description': ['Melanoma', 'Nevus (49%)', 'Basal Cell Carc.', 'Actinic Kerat.', 'Benign Kerat.', 'Dermatofibroma', 'Vascular Lesion', 'Squamous Cell Carc.'],
    'Recall': [f'{r:.1f}%' for r in recalls]
})
display(df_recs)

### 5. Confusion Matrix Visualization: The Minority Collapse
Shows how the uncalibrated model completely defaults to majority Nevi and misses 100% of Dermatofibroma (DF) and Vascular lesions (VASC).

In [ ]:
import matplotlib.pyplot as plt

cm = confusion_matrix(y_test, preds)

plt.figure(figsize=(8, 6))
plt.imshow(cm, interpolation='nearest', cmap=plt.cm.Blues)
plt.title('Baseline A0: Confusion Matrix\n(Notice 0 detections for DF and VASC)', fontsize=12, fontweight='bold')
plt.colorbar()
tick_marks = np.arange(len(CLASSES))
plt.xticks(tick_marks, CLASSES, rotation=45)
plt.yticks(tick_marks, CLASSES)
plt.ylabel('True Diagnosis', fontweight='bold')
plt.xlabel('Predicted Diagnosis', fontweight='bold')

thresh = cm.max() / 2.
for i in range(cm.shape[0]):
    for j in range(cm.shape[1]):
        plt.text(j, i, format(cm[i, j], 'd'),
                 ha='center', va='center',
                 color='white' if cm[i, j] > thresh else 'black')

plt.tight_layout()
plt.show()